# Import package

In [3]:
from bs4 import BeautifulSoup as bs     # parsing
from urllib.request import urlopen      # scrapping
import pandas as pd                     # exporting
from tenacity import retry, stop_after_attempt
import http.client

In [4]:
from io import StringIO
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.pdfpage import PDFPage
import os
import logging
from tqdm import tqdm
import csv
import re

In [13]:
address_1 = re.compile('(?:ONE|\d{1,4})?[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)
address_2 = re.compile('(?:ONE|\d{1,4})[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)

city = re.compile('[\w\s]{1,20}[,] \w{1,}?[\d\W]{1,11} [\w\s]{1,3}')
date = re.compile('(?:(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?\s+(?:of\s+)?(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)|(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)\s+(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?)(?:\,)?\s*(?:\d{4})?|[0-3]?\d[-\./][0-3]?\d[-\./]\d{2,4}', re.IGNORECASE)
number = re.compile('\d{1,20}')
title = ['PRESIDENT', 'VICE PRESIDENT', 'VICE PRESIDENT, TAX', 'TREASURER', 'ASSISTANT TREASURER', 'ASST. TREASURER', 'ASSISTANT CLERK', 'CLERK',
         'SECRETARY', 'ASSISTANT SECRETARY', 'CORPORATE SECRETARY', 'GENERAL SECRETARY', 'DIRECTOR', 'SECRETARY & DIRECTOR', 'MEDICAL DIRECTOR',
         'ASST TREAS/SECY', 'EDUCATION DIRECTOR', 'CO-WARDEN', 'VICE PRESIDENT FOOTBALL', 'VICE PRESIDENT CHEER', 'REGISTRAR', 'PUBLIC RELATIONS',
         'SENIOR PASTOR', 'ASSISTANT PASTOR', 'CHURCH LEADER', 'COLLECTOR', 'MODERATOR', 'CHAIR', 'GENERAL MEMBER', 'CEO', 'CFO',
         'GENERAL EVANGELIST', 'EXECUTIVE MINISTER', 'CORP CONTROLLER', 'CORPORATE CONTROLLER', 'OFFICER', 'CHAIRMAN & CEO', 'MANAGER',
         'PRESIDENT & CHIEF OPERATING', 'OPERATIONS', 'EDUCATION', 'CHAIRMAN', 'ICE DIRECTOR', 'CHIEF FINANCIAL OFFICER', 'DIRECTOR OF FINANCE',
         'CHIEF EXECUTIVE OFFICER', 'CHIEF OPERATING OFFICER', 'CHIEF COMPLIANCE OFFICER', 'IMMEDIATE PAST CHAIR', 'CHAIR-ELECT',
         'DIRECTOR-CHAIRMAN', 'CFO-TREASURER', 'EVP-COO', 'SENIOR MINISTER', 'MISSIONARY', 'ARCHIVIST']

# Util functions

In [3]:
def process_pdf(rsrcmgr, device, fp, pagenos=None, maxpages=0, password='', caching=True, check_extractable=True):
    interpreter = PDFPageInterpreter(rsrcmgr, device)
    for page in PDFPage.get_pages(fp, pagenos, maxpages=maxpages, password=password,
                                  caching=caching, check_extractable=check_extractable):
        interpreter.process_page(page)
    return

In [4]:
def read_pdf(pdf):
    rsrcmgr = PDFResourceManager()
    retstr = StringIO()
    laparams = LAParams()
    device = TextConverter(rsrcmgr, retstr, laparams=laparams)
    process_pdf(rsrcmgr, device, pdf)
    device.close()
    content = retstr.getvalue()
    retstr.close()
    lines = str(content).split("\n")
    return lines

In [5]:
def write_csv(content, path):
    with open(path, 'w') as f:
        w = csv.writer(f)
        w.writerows(content.items())

In [1]:
def handle(rst):
    idnum = ''
    exactName = ''
    jurisdiction = ''
    registeredName = ''
    registeredAddress = ''
    principalAddress = ''
    directors_info = []
    business = ''
    address = address_2
    rst = list(filter(lambda x: x != '', rst))
    
    directorsTitle = []
    directorsStreet = []
    directorsCity = []
    directorsName = []
    first = []
    last = []
    middle = []
    
    for i in range(len(rst)):
        if 'Identification Number' in rst[i]:
            idnum = rst[i].split(': ')[1]
#             idnum =  re.findall(r'.*: ([0-9]*)', rst[i])[0]
            
        if 'Exact name' in rst[i] or 'exact name' in rst[i]:
            exactName = rst[i+1]
            
        if 'principal office' in rst[i]:
            principalStreetList = []
            principalStreet = ''
            principalCity = ''
            principalState = ''
            principalZip = ''
            principalCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'directors' not in tmp:
                if tmp.isupper() and '3.' not in tmp:
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        principalStreetList.append(tmp.strip())
                    else:
                        principalStreetList.append(tmp.split(',')[0].strip())
                
                if 'State' in tmp:
                    try:
                        principalState = tmp.split(': ')[1].strip()
                    except:
                        principalState = ''
                if 'Zip' in tmp:
                    try:
                        principalZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        principalZip = ''
                if 'Country' in tmp:
                    principalCountry = tmp.split(': ')[1].strip()
                
                count += 1
                tmp = rst[count]
            
            for i in range(len(principalStreetList)):
                principalStreet += principalStreetList[i]
                principalStreet += ', '
            
            principalAddress = principalStreet + principalState + \
                               ' ' + principalZip + ' ' + principalCountry
                
        if 'corporation registered office' in rst[i]:
            registeredStreetList = []
            registeredStreet = ''
            registeredCity = ''
            registeredState = ''
            registeredZip = ''
            registeredCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'principal office' not in tmp and 'directors' not in tmp:
                if (address.search(tmp) != None) and (tmp.isupper()):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        registeredStreetList.append(tmp.strip())
                    else:
                        registeredStreetList.append(tmp.split(',')[0].strip())
                elif registeredName == '':
                    if tmp.isupper():
                        registeredName = tmp.strip()
                elif tmp.isupper() and tmp.replace(' ', '').replace('.', '').isalpha():
                    registeredCity = tmp.strip()
                    
                if 'State' in tmp:
                    try:
                        registeredState = tmp.split('State: ')[1].strip()
                    except:
                        registeredState = ''
                if 'Zip' in tmp:
                    try:
                        registeredZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        registeredZip = ''
                if 'Country' in tmp:
                    registeredCountry = tmp.split('Country: ')[1].strip()
                    
                count += 1
                tmp = rst[count]
            
            for i in range(len(registeredStreetList)):
                registeredStreet += registeredStreetList[i]
                registeredStreet += ', '
            
            registeredAddress = registeredStreet + registeredCity + ', ' + registeredState + \
                                ' ' + registeredZip + ' ' + registeredCountry
            
        if 'Title' in rst[i]:
            count = i
            tmp = rst[count]
            
            while 'Briefly describe' not in tmp and 'cemetery corporation' not in tmp and 'undersigned' not in tmp:
                if count != i and 'Title' in tmp:
                    break
                # sometimes the title name varies, we can just check if it contains any one in the title list
                if tmp.replace('\xa0', '').strip() in title or \
                bool([t for t in title if (t in tmp.replace('\xa0', '').strip())]): 
                    directorsTitle.append(tmp.split(": ")[1])
                elif ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp.upper()):
                    directorsCity.append(tmp.split(": ")[1])
                
                # usually the column of name is on the left of the address, so this one step should go first
                elif (tmp.isupper()) and (len(tmp.split()) > 1) and \
                     (re.sub(r'[^\w\s]','',tmp).replace('\xa0', '').replace(' ', '').isalpha()):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                
                elif 'FIRST' in tmp.upper():
                    first.append(tmp.split(": ")[1])

                elif 'MIDDLE' in tmp.upper():
                    middle.append(tmp.split(": ")[1])
                    
                elif 'Residential Address' in tmp:
                    directorsStreet.append(tmp.split(':')[1].strip())
                        
                elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp.upper()) or (tmp.upper().strip() == 'SAME'):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    elif ':' in tmp:
                        directorsStreet.append(tmp.split(':')[1].strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                    
                count += 1
                tmp = rst[count]
            
            if len(directorsCity) == 0:
                while 'Check' not in tmp:
                    if ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp.upper()):
                        directorsCity.append(tmp.split(": ")[1])
                    elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp.upper()):
                        directorsStreet.append(tmp.split(',')[0].strip())
                    count += 1
                    tmp = rst[count]
                
                
        if 'The general character' in rst[i] or 'Briefly describe' in rst[i]:
            count = i + 1
            tmp = rst[count]
            while 'cid' in tmp or tmp == ' ' or tmp == '' or 'service, the service to be rendered:' in tmp or \
                address.search(tmp) != None or city.search(tmp) != None or tmp.isupper() == False:
                count += 1
                tmp = rst[count]
                break
            business = tmp.strip()
            
        if 'LAST NAME' in rst[i].upper():
            last.append(rst[i].split(": ")[1])
            
    for i in range(len(last)):
        directorsName.append(first[i] + ' ' + middle[i] + ' ' + last[i])
               
    print(directorsCity)
    print(directorsTitle)
    print(directorsStreet)
    print(directorsName)
    directors_info = [[] for _ in range(len(directorsTitle))]

    if len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet) and \
        len(directorsStreet) == len(directorsCity):
        for i in range(len(directorsTitle)):
            directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                     directorsStreet[i] + ', ' + directorsCity[i])
    elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet):
        for i in range(len(directorsTitle)):
            directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                     directorsStreet[i])
    elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsCity):
        for i in range(len(directorsTitle)):
            directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + directorsCity[i])
    elif len(directorsStreet) == len(directorsName) and len(directorsName) == len(directorsCity):
        for i in range(len(directorsStreet)):
            directors_info[i].append(directorsName[i] + ', ' + \
                                     directorsStreet[i] + ', ' + directorsCity[i])
    elif len(directorsTitle) == len(directorsName):
        for i in range(len(directorsTitle)):
            directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i])
    elif len(directorsTitle) == len(directorsCity):
        for i in range(len(directorsTitle)):
            directors_info[i].append(directorsTitle[i] + ', ' + directorsCity[i])
    elif len(directorsTitle) == len(directorsStreet):
        for i in range(len(directorsTitle)):
            directors_info[i].append(directorsTitle[i] + ', ' + directorsStreet[i])
    elif len(directorsCity) == len(directorsName):
        for i in range(len(directorsName)):
            directors_info[i].append(directorsName[i] + ', ' + directorsCity[i])
    elif len(directorsCity) == len(directorsStreet):
        for i in range(len(directorsCity)):
            directors_info[i].append(directorsStreet[i] + ', ' + directorsCity[i])
    elif len(directorsName) == len(directorsStreet):
        for i in range(len(directorsName)):
            directors_info[i].append(directorsName[i] + ', ' + directorsStreet[i])

    try:
        id = format(int(idnum), '09d') if len(idnum) <= 9  and len(idnum) else None
    except:
        id = idnum
                    
    rec = {

        'ID Number': id, # fixed length = 9

        'Business Name': exactName,

        'Jurisdiction': jurisdiction,
        
        'Registered Office Name': registeredName,
        
        'Registered Office Address': registeredAddress,
        
        'Principal Office Address': principalAddress,
        
        'Directors': directors_info,
        
        'Business': business
        
        }
    
    return rec

TabError: inconsistent use of tabs and spaces in indentation (<ipython-input-1-b19e580e9643>, line 115)

In [2]:
path = '.\PDF_data\PDF'
dirs = os.listdir(path)
# business = {}
results = [] 
for file in tqdm(dirs):
    pdf = path + '/' + file
    with open(pdf, 'rb') as f:
        try:
            rst = read_pdf(f)
        except:
            continue
        try:
            info = handle(rst)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = format(int(file[:-4]), '09d')
            results.append(info)
        except IndexError:
            # print('list index out of range')
            continue
        except ValueError:
            info = handle(rst)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = file[:-4]
            results.append(info)
            continue

NameError: name 'os' is not defined

In [131]:
resultsDF = pd.DataFrame(results)
resultsDF.to_csv('~/Desktop/sec_results.csv')

In [6]:
result = pd.read_csv('~/Desktop/sec_results.csv')

In [9]:
path = './../Desktop/PDF_extracted' 
for file in tqdm(result['ID Number']):
    pdf = path + '/' + format(file, '09d') + '.pdf'
    if os.path.exists(pdf):
        shutil.move(pdf, './../Desktop/PDF_extracted' + '/' + format(file, '09d') + '.pdf')

100%|█████████████████████████████████████████████████████████████████████████████| 184/184 [00:00<00:00, 15333.53it/s]


In [11]:
resultDF = pd.read_csv('~/Desktop/results/sec_results1.csv')

In [12]:
result = pd.concat([result, resultDF], ignore_index=0)
result.to_csv('~/Desktop/sec_results1.csv')